# 🚀 Master Notebook: Huấn Luyện Tất Cả Các Agent Trên Kaggle Dual T4 (`32GB VRAM`)
Notebook này gọi lần lượt các script huấn luyện độc lập của từng Agent:
1. **`training_lab/agent1_retrieval/train_agent1_t4x2.py`** (Agent 1 Router QLoRA `r=64, DoRA`)
2. **`training_lab/agent3_summarizer/train_agent3_t4x2.py`** (Agent 3 Generator `SFT+DPO` & Critic `Hard-Negatives`)
3. **`training_lab/agent4_fact_checker/train_agent4_t4x2.py`** (Agent 4 Fact-Checker `3-Class NLI`)

*(Nếu chỉ muốn huấn luyện riêng lẻ 1 Agent, bạn có thể mở trực tiếp file `.ipynb` nằm bên trong thư mục của từng Agent tương ứng).*


In [ ]:
# BƯỚC 1: CẤU HÌNH GOOGLE DRIVE LINK HOẶC GITHUB TOKEN
GOOGLE_DRIVE_FOLDER_LINK = ""
COLAB_DRIVE_SAVE_PATH = "/content/drive/MyDrive/MultiAgent_RAG_Weights"
GITHUB_TOKEN = ""

RUN_AGENT_1 = True   # Agent 1: Retrieval Router
RUN_AGENT_3 = True   # Agent 3: Summarizer (Generator SFT+DPO & Critic)
RUN_AGENT_4 = True   # Agent 4: Fact-Checker NLI Verifier


In [ ]:
# BƯỚC 2: CLONE GITHUB & CÀI ĐẶT THƯ VIỆN
!pip install -q -U torch transformers peft trl bitsandbytes datasets accelerate sentencepiece gdown safetensors
import os, subprocess, torch
print(f"[GPU Check] Số lượng GPU: {torch.cuda.device_count()}")
PROJECT_DIR = "/content/Multi-Agent_RAG" if os.path.exists("/content") else "/kaggle/working/Multi-Agent_RAG"
if not os.path.exists(PROJECT_DIR):
    subprocess.run(["git", "clone", "https://github.com/ducmanh-jr/Multi-Agent_RAG.git", PROJECT_DIR], check=True)
os.chdir(PROJECT_DIR)


In [ ]:
# BƯỚC 3: CHẠY HUẤN LUYỆN TUẦN TỰ TỪNG AGENT (TỰ ĐỘNG XẢ 100% VRAM GIỮA CÁC AGENT)
if RUN_AGENT_1:
    !python training_lab/agent1_retrieval/train_agent1_t4x2.py --epochs 3
if RUN_AGENT_3:
    !python training_lab/agent3_summarizer/train_agent3_t4x2.py --part both --epochs 2
if RUN_AGENT_4:
    !python training_lab/agent4_fact_checker/train_agent4_t4x2.py --epochs 4


In [ ]:
# BƯỚC 4: TỰ ĐỘNG ĐỒNG BỘ TRỌNG SỐ VỀ GOOGLE DRIVE HOẶC GITHUB
if GITHUB_TOKEN:
    auth_repo = f"https://{GITHUB_TOKEN}@github.com/ducmanh-jr/Multi-Agent_RAG.git"
    subprocess.run(["git", "config", "user.email", "trainer@kaggle-t4x2.local"], check=False)
    subprocess.run(["git", "config", "user.name", "Kaggle T4x2 Auto Trainer"], check=False)
    subprocess.run(["git", "add", "training_lab/agent1_retrieval/weights", "training_lab/agent3_summarizer/weights", "training_lab/agent4_fact_checker/weights"], check=False)
    subprocess.run(["git", "commit", "-m", "feat(training_lab): update all per-agent T4x2 trained weights"], check=False)
    subprocess.run(["git", "push", auth_repo, "HEAD:main"], check=True)
    print("🎉 Đã tự động đẩy toàn bộ trọng số của các Agent về GitHub!")
